<h1> DATA Preprocessing <h1>

In [3]:
# ==========================================
# Phase 1: Load and Understand the Dataset
# ==========================================

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files


print("Please upload your 'customer_data_raw.xlsx' file:")
uploaded = files.upload()


file_name = list(uploaded.keys())[0]


excel_file = pd.ExcelFile(file_name)
print(f"\n Available Sheet Names: {excel_file.sheet_names}")


df = pd.read_excel(file_name, sheet_name=excel_file.sheet_names[0])
print(f"\n Dataset loaded successfully! Shape: {df.shape} (Rows: {df.shape[0]}, Columns: {df.shape[1]})")

Please upload your 'customer_data_raw.xlsx' file:


In [ ]:

print("--- FIRST 5 ROWS ---")
display(df.head())


print("\n--- LAST 5 ROWS ---")
display(df.tail())


print("\n--- DATASET INFORMATION ---")
df.info()

In [ ]:

print("--- NUMERICAL SUMMARY STATISTICS ---")
display(df.describe().T)


print("\n--- CATEGORICAL SUMMARY STATISTICS ---")
display(df.describe(include=['O', 'bool']).T)


print("\n--- COLUMN DATA TYPES BREAKDOWN ---")
print(df.dtypes.value_counts())

In [ ]:
print("--- An overview summary table for data hygiene evaluation ---")

data_health = pd.DataFrame({
    'Data_Type': df.dtypes,
    'Null_Count': df.isnull().sum(),
    'Null_Percentage (%)': (df.isnull().sum() / len(df)) * 100,
    'Unique_Values': df.nunique()
}).sort_values(by='Null_Percentage (%)', ascending=False)

print("--- DATA HEALTH OVERVIEW ---")
display(data_health)

In [ ]:
# ==========================================
# Phase 2: Data Cleaning
# ==========================================

<h2>Handle Missing Values<h2>

In [ ]:
# Check how many missing values each column has
missing_counts = df.isnull().sum()

# Display only columns that have missing values
missing_counts[missing_counts > 0]

In [ ]:
from sklearn.impute import SimpleImputer

# Create a copy so we don't mess up our main dataset
df_clean = df.copy()

# A. Mean Imputation for 'Age'
mean_imputer = SimpleImputer(strategy='mean')
df_clean['Age'] = mean_imputer.fit_transform(df_clean[['Age']])

# B. Median Imputation for 'Salary'
median_imputer = SimpleImputer(strategy='median')
df_clean['Salary'] = median_imputer.fit_transform(df_clean[['Salary']])

# C. Mode Imputation for 'Gender'
mode_imputer = SimpleImputer(strategy='most_frequent')
df_clean['Gender'] = mode_imputer.fit_transform(df_clean[['Gender']]).ravel()

# D. Constant Value Imputation for 'Review'
constant_imputer = SimpleImputer(strategy='constant', fill_value='No Review')
df_clean['Review'] = constant_imputer.fit_transform(df_clean[['Review']]).ravel()

print("Missing values after Simple Imputation:")
print(df_clean[['Age', 'Salary', 'Gender', 'Review']].isnull().sum())

In [ ]:
from sklearn.impute import KNNImputer

# Select numeric columns to use for KNN Imputation
knn_imputer = KNNImputer(n_neighbors=3)

# Impute missing values in 'MonthlySpend' using surrounding data
df_clean['MonthlySpend'] = knn_imputer.fit_transform(df_clean[['MonthlySpend']])

print("Missing values in MonthlySpend:", df_clean['MonthlySpend'].isnull().sum())

<h2>Remove Duplicate Records<h2>

In [ ]:
# 1. Check how many total duplicate rows exist
num_duplicates = df_clean.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

# 2. Remove duplicate rows and keep only the first occurrence
df_clean = df_clean.drop_duplicates()

print(f"Dataset shape after removing duplicates: {df_clean.shape}")

<h2>Fix Invalid Values<h2>

In [ ]:
# A. Fix Negative or Unrealistic Age (e.g., set negative ages to positive or NaN)
df_clean['Age'] = df_clean['Age'].apply(lambda x: abs(x) if x < 0 else x)

# B. Fix Invalid Salary (e.g., set negative salary to NaN or lower bound)
df_clean['Salary'] = df_clean['Salary'].apply(lambda x: np.nan if x <= 0 else x)

# C. Fix Impossible Dates (replace bad date placeholders like '9999-99-99' with NaN)
df_clean['JoinDate'] = df_clean['JoinDate'].replace(['9999-99-99', 'invalid'], np.nan)

print("Invalid values cleaned!")

<h2>Standardize Categorical Values<h2>

In [ ]:
# Standardize 'Gender' values to consistent labels
df_clean['Gender'] = df_clean['Gender'].astype(str).str.lower().str.strip()

# Map all variations to standard categories
gender_mapping = {
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female'
}

df_clean['Gender'] = df_clean['Gender'].map(gender_mapping).fillna('Other')

print("Unique values in Gender column after standardization:")
print(df_clean['Gender'].value_counts())

<h2>Convert Data Types<h2>

In [ ]:
# A. Convert numeric text columns to clean numbers
df_clean['SalaryText'] = pd.to_numeric(df_clean['SalaryText'], errors='coerce')

# B. Convert string booleans ('TRUE'/'FALSE' or 'Yes'/'No') into Python Booleans (True/False)
boolean_mapping = {'True': True, 'False': False, 'Yes': True, 'No': False, True: True, False: False}
df_clean['Remote'] = df_clean['Remote'].map(boolean_mapping)

# C. Convert Date string column to proper Datetime format
df_clean['JoinDate'] = pd.to_datetime(df_clean['JoinDate'], errors='coerce', format='mixed')

print(df_clean[['SalaryText', 'Remote', 'JoinDate']].dtypes)

<h2>Parse Date Columns<h2>

In [ ]:
# Extract date features from 'JoinDate'
df_clean['JoinDate_Year'] = df_clean['JoinDate'].dt.year
df_clean['JoinDate_Month'] = df_clean['JoinDate'].dt.month
df_clean['JoinDate_Day'] = df_clean['JoinDate'].dt.day
df_clean['JoinDate_Weekday'] = df_clean['JoinDate'].dt.weekday  # Monday=0, Sunday=6
df_clean['JoinDate_Quarter'] = df_clean['JoinDate'].dt.quarter

# Preview the newly generated date features
df_clean[['JoinDate', 'JoinDate_Year', 'JoinDate_Month', 'JoinDate_Day', 'JoinDate_Weekday', 'JoinDate_Quarter']].head()

In [ ]:
df_clean


In [ ]:
# ==========================================
# Phase 3: Categorical Encoding
# ==========================================

<h2>Label Encoding<h2>

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a copy for Phase 3
df_encoded = df_clean.copy()

# Initialize LabelEncoder
label_enc = LabelEncoder()

# Encode binary target column 'Purchased' ('Yes' -> 1, 'No' -> 0)
df_encoded['Purchased_Encoded'] = label_enc.fit_transform(df_encoded['Purchased'])

print("--- Label Encoding Results ---")
print(df_encoded[['Purchased', 'Purchased_Encoded']].head())

<h2>Ordinal Encoding<h2>

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Define explicit order for categorical features
education_order = ['High School', 'Bachelor', 'Master', 'PhD']
job_level_order = ['Junior', 'Mid', 'Senior']
loyalty_order = ['Bronze', 'Silver', 'Gold', 'Platinum']

# 2. Handle missing values in these columns before encoding (fill with most frequent or a placeholder)
df_encoded['Education'] = df_encoded['Education'].fillna('High School')

# 3. Create Ordinal Encoder with explicit categories
ordinal_enc = OrdinalEncoder(categories=[education_order, job_level_order, loyalty_order])

# 4. Apply transformation
encoded_cols = ordinal_enc.fit_transform(df_encoded[['Education', 'JobLevel', 'LoyaltyTier']])

# Assign back to DataFrame
df_encoded[['Education_Ord', 'JobLevel_Ord', 'LoyaltyTier_Ord']] = encoded_cols

print("--- Ordinal Encoding Results ---")
print(df_encoded[['Education', 'Education_Ord', 'JobLevel', 'JobLevel_Ord', 'LoyaltyTier', 'LoyaltyTier_Ord']].head())

<h2>One-Hot Encoding (Unordered Categories)<h2>


In [ ]:
# Using pandas get_dummies for simple, clean one-hot encoding
departments_ohe = pd.get_dummies(df_encoded['Department'], prefix='Dept', dtype=int)

# Inspect the created columns
print("--- One-Hot Encoding Columns Created ---")
print(departments_ohe.head())

# Concatenate back to main DataFrame
df_encoded = pd.concat([df_encoded, departments_ohe], axis=1)

In [ ]:
# ==========================================
# Phase 4: Numerical Preprocessing
# ==========================================

<h2>Handle Outliers<h2>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mstats

df_num = df_clean.copy()

# A. IQR Capping Function
def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Cap values below lower_bound and above upper_bound
    df[column] = np.clip(df[column], lower_bound, upper_bound)
    return df

# Apply IQR Capping on OutlierFeature
df_num = cap_outliers_iqr(df_num, 'OutlierFeature')

# B. Winsorization (Caps top 5% and bottom 5% values)
df_num['Salary_Winsorized'] = mstats.winsorize(df_num['Salary'], limits=[0.05, 0.05])

print("Outliers capped successfully!")

<h2>Feature Scaling<h2>

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# 1. StandardScaler
std_scaler = StandardScaler()
df_num['Age_Standard'] = std_scaler.fit_transform(df_num[['Age']])

# 2. MinMaxScaler
minmax_scaler = MinMaxScaler()
df_num['Age_MinMax'] = minmax_scaler.fit_transform(df_num[['Age']])

# 3. RobustScaler
robust_scaler = RobustScaler()
df_num['Age_Robust'] = robust_scaler.fit_transform(df_num[['Age']])

print("--- Scaling Comparison for 'Age' ---")
print(df_num[['Age', 'Age_Standard', 'Age_MinMax', 'Age_Robust']].head())

<h2>Feature Transformation (Fixing Skewness)<h2>

In [ ]:
from sklearn.preprocessing import FunctionTransformer, PowerTransformer

# A. Log Transformation (np.log1p avoids log(0) errors)
log_transformer = FunctionTransformer(np.log1p, validate=True)
df_num['SkewedIncome_Log'] = log_transformer.fit_transform(df_num[['SkewedIncome']])

# B. Box-Cox Transformation (requires strictly positive numbers > 0)
boxcox_pt = PowerTransformer(method='box-cox')
df_num['SkewedIncome_BoxCox'] = boxcox_pt.fit_transform(df_num[['SkewedIncome']])

# C. Yeo-Johnson Transformation (works with 0 and negative values too)
yeojohnson_pt = PowerTransformer(method='yeo-johnson')
df_num['SkewedIncome_YeoJohnson'] = yeojohnson_pt.fit_transform(df_num[['SkewedIncome']])

print("--- Skewness Values Before & After ---")
print(f"Original Skewness   : {df_num['SkewedIncome'].skew():.2f}")
print(f"Log Transformed     : {df_num['SkewedIncome_Log'].skew():.2f}")
print(f"Box-Cox Transformed : {df_num['SkewedIncome_BoxCox'].skew():.2f}")